<!-- bootcamp-header: generated by tools/build_headers.py, edit the README timetable instead -->
# Solutions: Project work

Worked solutions for the session [`12_W3_Mon_Larger_Project.ipynb`](https://github.com/LemmensJens/dtaantwerp26-27.github.io/blob/DTA_Bootcamp_2026_students/notebooks/12_W3_Mon_Larger_Project.ipynb).  

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LemmensJens/dtaantwerp26-27.github.io/blob/DTA_Bootcamp_2026_students/exercises/solutions/12_SOL_Larger_Project.ipynb)

One possible solution, following the five steps of the session notebook. Other designs are equally valid; what matters is that each function does one thing, is tested on its own, and that the program meets every requirement of the assignment.

## The project: an authentication system

**Assignment.** Write a small, naive authentication system, of the kind a website uses to let people register and log in. Before you read the detailed requirements below, think for a minute about what such a system must be able to do, and sketch it on paper: what does it ask the user, what does it store, what can go wrong, and what should happen then?

The system needs two parts.

**(A) Registration.** Ask the user, with `input()`, for an e-mail address (which serves as the user name) and a password, and store the pair in a suitable data structure, the "database". Only accept the registration if both are valid:

- A **valid e-mail address**, for our purposes: contains exactly one `@`; ends in `.com`, `.org` or `.be`; contains no punctuation other than `-`, `@` and `.`.
- A **valid password**: at least 8 characters long; contains upper case *and* lower case letters; contains at least two digits; contains no punctuation.

Keep asking until the user gets it right. Print exactly these messages under the corresponding conditions:

- `You entered an invalid email address. Try again.`
- `That user name has already been taken. Try again.`
- `Your password is not valid. Try again.` (feel free to say what was wrong with it)
- `Your account has been successfully created!`

**(B) Login.** Ask the user for an e-mail address and a password, and grant access if the address is in the database and the password matches. A wrong password costs an attempt; after three wrong passwords, stop and "send a warning e-mail" to the last address used (we cannot really send e-mail: print a message that says what the e-mail would contain). Messages:

- `This email address does not exist in our database. Try again.`
- `Your email address exists, but your password is incorrect. Try again. (N attempts left)`
- `You have successfully logged in!`
- `You have exhausted your 3 login attempts. A warning message has been emailed to the last email address used.`

### Step 1: validating an e-mail address

In [1]:
import string

ALLOWED_ENDINGS = ('.com', '.org', '.be')
ALLOWED_PUNCTUATION = '-@.'


def is_valid_email(address):
    """Return True if address has exactly one @, an allowed ending and no other punctuation than - @ and ."""
    if address.count('@') != 1:
        return False
    if not address.endswith(ALLOWED_ENDINGS):
        return False
    for char in address:
        if char in string.punctuation and char not in ALLOWED_PUNCTUATION:
            return False
    return True


print(is_valid_email('ada.lovelace@example.com'), 'expected True')
print(is_valid_email('ada-lovelace@example.be'), 'expected True')
print(is_valid_email('ada@@example.com'), 'expected False')
print(is_valid_email('ada@example.nl'), 'expected False')
print(is_valid_email('ada_lovelace@example.com'), 'expected False')

True expected True
True expected True
False expected False
False expected False
False expected False


### Step 2: validating a password

In [2]:
def is_valid_password(password):
    """Return True if password is at least 8 characters long, has upper and lower case letters,
    at least two digits and no punctuation."""
    if len(password) < 8:
        return False
    uppercase = 0
    lowercase = 0
    digits = 0
    for char in password:
        if char in string.punctuation:
            return False
        if char.isupper():
            uppercase += 1
        if char.islower():
            lowercase += 1
        if char.isdigit():
            digits += 1
    return uppercase > 0 and lowercase > 0 and digits >= 2


print(is_valid_password('Wonderland1865'), 'expected True')
print(is_valid_password('wonderland1865'), 'expected False, no upper case')
print(is_valid_password('Wonder1'), 'expected False, too short')
print(is_valid_password('Wonderland!1865'), 'expected False, punctuation')
print(is_valid_password('Wonderland1'), 'expected False, one digit')

True expected True
False expected False, no upper case
False expected False, too short
False expected False, punctuation
False expected False, one digit


### Step 3: registration

In [3]:
def register(users):
    """Ask for a new e-mail address and password until both are valid, and add them to users."""
    while True:
        address = input('Choose a user name (your e-mail address): ')
        if not is_valid_email(address):
            print('You entered an invalid email address. Try again.')
        elif address in users:
            print('That user name has already been taken. Try again.')
        else:
            break
    while True:
        password = input('Choose a password: ')
        if is_valid_password(password):
            break
        print('Your password is not valid. Try again. (At least 8 characters, upper and lower case, two digits, no punctuation.)')
    users[address] = password
    print('Your account has been successfully created!')

In [ ]:
users = {}
register(users)
print(users)

### Step 4: login

In [4]:
def send_warning(address):
    """Pretend to send a warning e-mail (we only print it)."""
    print(f'--- e-mail to {address} ---')
    print('Someone tried to log in to your account three times with a wrong password.')
    print('If this was not you, please change your password.')
    print('---')


def login(users, max_attempts=3):
    """Ask for credentials; return True on success, False after max_attempts wrong passwords."""
    attempts_left = max_attempts
    while attempts_left > 0:
        address = input('User name: ')
        if address not in users:
            print('This email address does not exist in our database. Try again.')
            continue
        password = input('Password: ')
        if password == users[address]:
            print('You have successfully logged in!')
            return True
        attempts_left -= 1
        print(f'Your email address exists, but your password is incorrect. Try again. ({attempts_left} attempts left)')
    print(f'You have exhausted your {max_attempts} login attempts. A warning message has been emailed to the last email address used.')
    send_warning(address)
    return False

In [ ]:
login(users)

### Step 5: a menu

In [ ]:
def main(users):
    """Offer to register, log in or quit, until the user quits."""
    while True:
        choice = input('Do you want to (r)egister, (l)og in or (q)uit? ')
        if choice == 'r':
            register(users)
        elif choice == 'l':
            login(users)
        elif choice == 'q':
            print('Goodbye.')
            break
        else:
            print('Please answer r, l or q.')


main(users)

### Extension: persistence

The database as a tab-separated text file, one user per line. `load_users()` returns an empty dictionary when the file does not exist yet, so that the program also works on the very first run.

In [5]:
import os


def save_users(users, path):
    """Write users to path, one 'address<TAB>password' line per user."""
    with open(path, 'w', encoding='utf-8') as outfile:
        for address, password in users.items():
            outfile.write(f'{address}\t{password}\n')


def load_users(path):
    """Read the users written by save_users(); an empty dictionary if the file does not exist."""
    users = {}
    if not os.path.exists(path):
        return users
    with open(path, encoding='utf-8') as infile:
        for line in infile:
            address, password = line.strip().split('\t')
            users[address] = password
    return users


test_users = {'ada.lovelace@example.com': 'Analytical1843', 'charles.babbage@example.org': 'Difference1822'}
save_users(test_users, 'users.txt')
print(load_users('users.txt'))
print(load_users('users.txt') == test_users)

{'ada.lovelace@example.com': 'Analytical1843', 'charles.babbage@example.org': 'Difference1822'}
True


### Extension: better feedback

`password_problems()` returns the list of rules that were broken. An empty list is *falsy*, so `if not problems:` means "the password is fine": the same truthiness rule as for empty strings in the session on `if` and `else`.

In [6]:
def password_problems(password):
    """Return a list describing every rule that password breaks (empty if it is valid)."""
    problems = []
    if len(password) < 8:
        problems.append('at least 8 characters')
    if not [char for char in password if char.isupper()]:
        problems.append('an upper case letter')
    if not [char for char in password if char.islower()]:
        problems.append('a lower case letter')
    if len([char for char in password if char.isdigit()]) < 2:
        problems.append('at least two digits')
    if [char for char in password if char in string.punctuation]:
        problems.append('no punctuation')
    return problems


for candidate in ['Wonderland1865', 'wonder', 'Wonderland!1']:
    problems = password_problems(candidate)
    if not problems:
        print(f'{candidate}: fine')
    else:
        print(f'{candidate}: needs ' + ', '.join(problems))

Wonderland1865: fine
wonder: needs at least 8 characters, an upper case letter, at least two digits
Wonderland!1: needs at least two digits, no punctuation
